In [1]:
import sys, os
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)
    
from data.nuscenes_data import NuscenesData
from data.nuscenes_wrapper import SparseCLIP_Dataset
from nuscenes.nuscenes import NuScenes
from pathlib import Path
import torch

In [ ]:
# Load dataset
data_path = Path("/home/ximeng/Dataset/nuscenes_full_v1_0/")
nusc = NuScenes(version='v1.0-trainval', dataroot=data_path)
is_train = 1
pre_frame = 0
future_frame = 0
dataset = NuscenesData(nusc, is_train, pre_frame, future_frame)

Loading NuScenes tables for version v1.0-trainval...
23 category,
8 attribute,
4 visibility,
64386 instance,
12 sensor,
10200 calibrated_sensor,
2631083 ego_pose,
68 log,
850 scene,
34149 sample,
2631083 sample_data,
1166187 sample_annotation,
4 map,
Done loading in 18.982 seconds.
Reverse indexing ...
Done reverse indexing in 4.8 seconds.


In [5]:
def sliding_window(seq, window_size, stride):
    L = len(seq)
    windows = []
    for start in range(0, L - window_size + 1, stride):
        windows.append(seq[start:start + window_size])
    if L % stride != 0 and L > 0:
        windows.append(seq[-window_size:])
    return windows

def pad_text_lidar_window(batch_pairs, window_size=32, stride=32):
    pts_dim = 1024
    all_windows = []

    # Create sliding windows for each sequence in the batch
    for pairs in batch_pairs:
        if len(pairs) == 0:
            continue
        windows = sliding_window(pairs, window_size, stride)
        all_windows.extend(windows)

    B_new = len(all_windows)
    padded_pts = torch.zeros(B_new, window_size, pts_dim, 3, dtype=torch.float32)
    mask = torch.zeros(B_new, window_size, dtype=torch.bool)
    padded_labels = [[''] * window_size for _ in range(B_new)]
    flat_labels = []

    for i, window in enumerate(all_windows):
        for j, (label, pts) in enumerate(window):
            if pts.shape != (pts_dim, 3):
                raise ValueError(f"Expected pts shape ({pts_dim}, 3), got {pts.shape}")
            padded_pts[i, j] = pts
            padded_labels[i][j] = label
            mask[i, j] = True
            flat_labels.append(label)

    return padded_pts, padded_labels, mask, flat_labels

In [4]:
def custom_collate_fn(batch):
    collated = {}
    keys = batch[0].keys()
    for k in keys:
        values = [b[k] for b in batch]
        if k == 'text_lidar_pair':
            values = [v for v in values if len(v) > 0]
            # Handle empty case
            if len(values) == 0:
                return None # Skip this batch
            # Use sliding window
            pts, labels, mask, flat_labels = pad_text_lidar_window(values, window_size=5, stride=5)
            collated['text_lidar_pts'] = pts
            collated['text_lidar_labels'] = labels
            collated['text_lidar_mask'] = mask
            collated['text_lidar_flat_labels'] = flat_labels
        else:
            # Use default collate for other keys
            try:
                collated[k] = torch.utils.data._utils.collate.default_collate(values)
            except Exception:
                collated[k] = values
    return collated

In [6]:
SparseCLIP_dataset = SparseCLIP_Dataset(dataset)
SparseCLIP_train_dataloader = torch.utils.data.DataLoader(SparseCLIP_dataset, batch_size=2, shuffle=True, collate_fn=custom_collate_fn)

ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
for sample in SparseCLIP_train_dataloader:
    text_lidar_pair = sample['text_lidar_pts']
    text_lidar_labels = sample['text_lidar_labels']
    text_lidar_mask = sample['text_lidar_mask']
    text_lidar_flat_labels = sample['text_lidar_flat_labels']
    break  # Just to test the data loading

In [24]:
print(text_lidar_pair.shape)
print(text_lidar_labels)
print(text_lidar_mask)
print(text_lidar_flat_labels)

torch.Size([5, 5, 1024, 3])
[['truck', 'motorcycle', 'car', 'pedestrian', 'barrier'], ['barrier', 'trafficcone', 'barrier', 'construction', 'car'], ['barrier', 'truck', 'car', 'car', 'barrier'], ['car', 'barrier', 'barrier', 'barrier', 'pedestrian'], ['bus', 'pedestrian', 'motorcycle', '', '']]
tensor([[ True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True,  True,  True],
        [ True,  True,  True, False, False]])
['truck', 'motorcycle', 'car', 'pedestrian', 'barrier', 'barrier', 'trafficcone', 'barrier', 'construction', 'car', 'barrier', 'truck', 'car', 'car', 'barrier', 'car', 'barrier', 'barrier', 'barrier', 'pedestrian', 'bus', 'pedestrian', 'motorcycle']
